In [ ]:
## 월간동향 다운로드

import re
import time
from pathlib import Path
from urllib.parse import unquote
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import requests
from bs4 import BeautifulSoup

# --- 설정 및 변수 ---
LIST_URL = (
    "https://nkinfo.unikorea.go.kr/nkp/report/list.do?pstgSeCode=ARGUMENT_MNTHNG&menuId=ARGUMENT_83"
)
DOWNLOAD_URL = "https://nkinfo.unikorea.go.kr/nkp/cmmn/FileDown.do"

HEADERS = {
    "Origin": "https://nkinfo.unikorea.go.kr",
    "Referer": LIST_URL,
    "User-Agent": "Mozilla/5.0",
}

OUTPUT_DIR = Path("downloads")
OUTPUT_DIR.mkdir(exist_ok=True)


# --- 함수 정의 ---
def repair_filename(filename: str) -> str:
    """레거시 HTTP 헤더의 한글 파일명 깨짐을 가능한 범위에서 복원한다."""
    for encoding in ("cp949", "euc-kr", "UTF-8"):
        try:
            return filename.encode("latin-1").decode(encoding)
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return filename


def get_filename(content_disposition: str, default_name: str) -> str:
    """Content-Disposition에서 안전한 저장 파일명을 반환한다."""
    if not content_disposition:
        return default_name

    # 표준 형식: filename*=UTF-8''%EC%A3%BC...
    match = re.search(
        r"filename\*\s*=\s*([^']*)''([^;]+)",
        content_disposition,
        flags=re.IGNORECASE,
    )
    if match:
        charset = match.group(1) or "utf-8"
        encoded_filename = match.group(2).strip().strip('"')

        try:
            filename = unquote(encoded_filename, encoding=charset)
        except (LookupError, UnicodeDecodeError):
            filename = unquote(encoded_filename)

        return Path(filename).name

    # 일반 형식: filename="..."
    match = re.search(
        r'filename\s*=\s*"?([^";]+)"?',
        content_disposition,
        flags=re.IGNORECASE,
    )
    if match:
        filename = match.group(1).strip()
        return Path(repair_filename(filename)).name

    return default_name


# --- 메인 실행 로직 ---
# 수집할 페이지 범위 설정 (예: 1페이지 ~ 15페이지)
page_num_list = [i for i in range(1,26)]

# 브라우저 옵션 설정
options = Options()
options.add_argument("--window-size=1920,1080")

# 1. 브라우저 실행 및 초기 1페이지 접속
driver = webdriver.Chrome(options=options)
driver.get(LIST_URL)
time.sleep(3)  # 최초 페이지 데이터가 로딩될 때까지 대기
# 2. 파일 다운로드를 위한 세션(Session) 유지
data = []

# 2. 파일 다운로드를 위한 세션(Session) 유지
with requests.Session() as session:
    for page_num in page_num_list:
        try:
            print(f"\n========== [ {page_num} 페이지 수집 시작 ] ==========")
            
            if page_num > 1:
                driver.execute_script(f'pageSub({page_num});')
                time.sleep(3)  # 자바스크립트가 새 데이터를 화면에 그릴 때까지 대기

            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")

            # 3. 게시판 행(tr) 단위로 순회하며 데이터 수집 및 다운로드
            for tr in soup.select("tbody > tr"):
                td_list = tr.find_all("td")
                
                if len(td_list) < 4:
                    continue
                
                # 텍스트 정보 추출
                num = td_list[0].text.strip()
                name = td_list[1].text.strip()
                regi_date = td_list[2].text.strip()
                
                # 첨부파일 정보 추출
                file_tags = tr.select("a.attachFile")
                saved_filenames=[]
                for file_tag in file_tags:
                    bbs_id = file_tag.get("data-trend-report-no") if file_tag else None
                    file_no = file_tag.get("data-file-no") if file_tag else None
                    

                    # 4. 첨부파일이 존재하는 경우에만 다운로드 로직 실행
                    if bbs_id and file_no:
                        payload = {
                            "bbsType": "report",
                            "bbsId": bbs_id,
                            "fileNo": file_no,
                        }
                    else:
                        print('파일 번호나 bbs_id가 존재하지 않습니다.')
                        continue
                        
                    try:
                        response = session.post(
                            DOWNLOAD_URL,
                            data=payload,
                            headers=HEADERS,
                            timeout=10,
                        )
                        response.raise_for_status()

                        content_type = response.headers.get("Content-Type", "").lower()
                        content_disposition = response.headers.get("Content-Disposition", "")

                        if "text/html" in content_type:
                            raise ValueError("파일 대신 HTML 응답을 받았습니다.")

                        default_name = f"report_{bbs_id}_file_{file_no}.hwp"
                        filename = get_filename(content_disposition, default_name)

                        if not Path(filename).suffix:
                            filename += ".hwp"

                        output_path = OUTPUT_DIR / filename
                        output_path.write_bytes(response.content)

                        print(f"[완료] 번호={num}, 파일={output_path.name}")
                        
                        # 성공적으로 저장된 파일명을 변수에 덮어씌움
                        saved_filename = output_path.name
                        
                        saved_filenames.append({'bsId': payload['bbsId'],
                                                '파일명': saved_filename})
                    except Exception as e:
                        print(f"[다운로드 실패] 번호={num}, 사유: {e}")
                        saved_filename = "다운로드 실패"
                    
                    # 서버 부하 방지를 위해 다운로드 건마다 휴식
                    time.sleep(0.5)
                final_filenames_str = ", ".join([item['파일명'] for item in saved_filenames]) if saved_filenames else "첨부파일 없음"
                
                # 5. 최종 데이터 리스트에 적재 (파일 유무와 상관없이 무조건 1행 추가)
                data.append({
                    '게시글_번호': num,
                    '게시글_명': name,
                    '게시일자': regi_date,
                    '파일명': final_filenames_str
                })
                
        except Exception as e:    
            print(f"페이지 {page_num} 실행 중 예상치 못한 에러 발생: {e}")

# 모든 작업 완료 후 브라우저 안전하게 종료
driver.quit()

# 수집된 데이터 확인해보기 (데이터 분석 준비 완료!)
import pandas as pd
df_supplementary = pd.DataFrame(data)
print("\n=== 최종 수집 결과 ===")
print(df_supplementary.head())


========== [ 1 페이지 수집 시작 ] ==========
[완료] 번호=249, 파일=월간북한동향(2026년 8월).hwpx
[완료] 번호=249, 파일=월간북한동향(2026년 8월).pdf
[완료] 번호=248, 파일=월간북한동향(2026년 7월).hwpx
[완료] 번호=248, 파일=월간북한동향(2026년 7월).pdf
[완료] 번호=247, 파일=월간북한동향(2026년 6월).hwpx
[완료] 번호=247, 파일=월간북한동향(2026년 6월).pdf
[완료] 번호=246, 파일=월간북한동향(2026년 5월).hwp
[완료] 번호=246, 파일=월간북한동향(2026년 5월).pdf
[완료] 번호=245, 파일=월간북한동향(2026년 4월).hwp
[완료] 번호=245, 파일=월간북한동향(2026년 4월).pdf
[완료] 번호=244, 파일=월간북한동향(2026년 3월).hwp
[완료] 번호=244, 파일=월간북한동향(2026년 3월).pdf
[완료] 번호=243, 파일=월간북한동향(2026년 2월).hwp
[완료] 번호=243, 파일=월간북한동향(2026년 2월).pdf
[완료] 번호=242, 파일=월간북한동향(2026년 1월).hwp
[완료] 번호=242, 파일=월간북한동향(2026년 1월).pdf
[완료] 번호=241, 파일=월간북한동향(2025년 12월).hwp
[완료] 번호=241, 파일=월간북한동향(2025년 12월).pdf
[완료] 번호=240, 파일=월간북한동향(2025년 11월).hwp
[완료] 번호=240, 파일=월간북한동향(2025년 11월).pdf

========== [ 2 페이지 수집 시작 ] ==========
[완료] 번호=239, 파일=월간북한동향(2025년 10월).hwp
[완료] 번호=239, 파일=월간북한동향(2025년 10월).pdf
[완료] 번호=238, 파일=월간북한동향(2025년 9월).hwp
[완료] 번호=238, 파일=월간북한동향(2025년 9월).pdf
[완료] 번호=237, 파일=월간북한동향(20

In [7]:
df_supplementary.to_excel('월간북한동향_파일명세.xlsx')

In [ ]:
## 주간동향 다운로드


import re
import time
from pathlib import Path
from urllib.parse import unquote
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import requests
from bs4 import BeautifulSoup

# --- 설정 및 변수 ---
LIST_URL = (
    "https://nkinfo.unikorea.go.kr/nkp/report/list.do"
    "?pstgSeCode=ARGUMENT_WIK&menuId=ARGUMENT_82"
)
DOWNLOAD_URL = "https://nkinfo.unikorea.go.kr/nkp/cmmn/FileDown.do"

HEADERS = {
    "Origin": "https://nkinfo.unikorea.go.kr",
    "Referer": LIST_URL,
    "User-Agent": "Mozilla/5.0",
}

OUTPUT_DIR = Path("downloads")
OUTPUT_DIR.mkdir(exist_ok=True)


# --- 함수 정의 ---
def repair_filename(filename: str) -> str:
    """레거시 HTTP 헤더의 한글 파일명 깨짐을 가능한 범위에서 복원한다."""
    for encoding in ("cp949", "euc-kr", "UTF-8"):
        try:
            return filename.encode("latin-1").decode(encoding)
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return filename


def get_filename(content_disposition: str, default_name: str) -> str:
    """Content-Disposition에서 안전한 저장 파일명을 반환한다."""
    if not content_disposition:
        return default_name

    # 표준 형식: filename*=UTF-8''%EC%A3%BC...
    match = re.search(
        r"filename\*\s*=\s*([^']*)''([^;]+)",
        content_disposition,
        flags=re.IGNORECASE,
    )
    if match:
        charset = match.group(1) or "utf-8"
        encoded_filename = match.group(2).strip().strip('"')

        try:
            filename = unquote(encoded_filename, encoding=charset)
        except (LookupError, UnicodeDecodeError):
            filename = unquote(encoded_filename)

        return Path(filename).name

    # 일반 형식: filename="..."
    match = re.search(
        r'filename\s*=\s*"?([^";]+)"?',
        content_disposition,
        flags=re.IGNORECASE,
    )
    if match:
        filename = match.group(1).strip()
        return Path(repair_filename(filename)).name

    return default_name


# --- 메인 실행 로직 ---
# 수집할 페이지 범위 설정 (예: 1페이지 ~ 15페이지)
page_num_list = [147,148]

# 브라우저 옵션 설정
options = Options()
options.add_argument("--window-size=1920,1080")

# 1. 브라우저 실행 및 초기 1페이지 접속
driver = webdriver.Chrome(options=options)
driver.get(LIST_URL)
time.sleep(3)  # 최초 페이지 데이터가 로딩될 때까지 대기
# 2. 파일 다운로드를 위한 세션(Session) 유지
data = []

# 2. 파일 다운로드를 위한 세션(Session) 유지
with requests.Session() as session:
    for page_num in page_num_list:
        try:
            print(f"\n========== [ {page_num} 페이지 수집 시작 ] ==========")
            
            if page_num > 1:
                driver.execute_script(f'pageSub({page_num});')
                time.sleep(3)  # 자바스크립트가 새 데이터를 화면에 그릴 때까지 대기

            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")

            # 3. 게시판 행(tr) 단위로 순회하며 데이터 수집 및 다운로드
            for tr in soup.select("tbody > tr"):
                td_list = tr.find_all("td")
                
                if len(td_list) < 4:
                    continue
                
                # 텍스트 정보 추출
                num = td_list[0].text.strip()
                name = td_list[1].text.strip()
                regi_date = td_list[2].text.strip()
                
                # 첨부파일 정보 추출
                file_tags = tr.select("a.attachFile")
                saved_filenames=[]
                for file_tag in file_tags:
                    bbs_id = file_tag.get("data-trend-report-no") if file_tag else None
                    file_no = file_tag.get("data-file-no") if file_tag else None
                    

                    # 4. 첨부파일이 존재하는 경우에만 다운로드 로직 실행
                    if bbs_id and file_no:
                        payload = {
                            "bbsType": "report",
                            "bbsId": bbs_id,
                            "fileNo": file_no,
                        }
                    else:
                        print('파일 번호나 bbs_id가 존재하지 않습니다.')
                        continue
                        
                    try:
                        response = session.post(
                            DOWNLOAD_URL,
                            data=payload,
                            headers=HEADERS,
                            timeout=10,
                        )
                        response.raise_for_status()

                        content_type = response.headers.get("Content-Type", "").lower()
                        content_disposition = response.headers.get("Content-Disposition", "")

                        if "text/html" in content_type:
                            raise ValueError("파일 대신 HTML 응답을 받았습니다.")

                        default_name = f"report_{bbs_id}_file_{file_no}.hwp"
                        filename = get_filename(content_disposition, default_name)

                        if not Path(filename).suffix:
                            filename += ".hwp"

                        output_path = OUTPUT_DIR / filename
                        output_path.write_bytes(response.content)

                        print(f"[완료] 번호={num}, 파일={output_path.name}")
                        
                        # 성공적으로 저장된 파일명을 변수에 덮어씌움
                        saved_filename = output_path.name
                        
                        saved_filenames.append({'bsId': payload['bbsId'],
                                                '파일명': saved_filename})
                    except Exception as e:
                        print(f"[다운로드 실패] 번호={num}, 사유: {e}")
                        saved_filename = "다운로드 실패"
                    
                    # 서버 부하 방지를 위해 다운로드 건마다 휴식
                    time.sleep(0.5)
                final_filenames_str = ", ".join([item['파일명'] for item in saved_filenames]) if saved_filenames else "첨부파일 없음"
                
                # 5. 최종 데이터 리스트에 적재 (파일 유무와 상관없이 무조건 1행 추가)
                data.append({
                    '게시글_번호': num,
                    '게시글_명': name,
                    '게시일자': regi_date,
                    '파일명': final_filenames_str
                })
                
        except Exception as e:    
            print(f"페이지 {page_num} 실행 중 예상치 못한 에러 발생: {e}")

# 모든 작업 완료 후 브라우저 안전하게 종료
driver.quit()

# 수집된 데이터 확인해보기 (데이터 분석 준비 완료!)
import pandas as pd
df_supplementary = pd.DataFrame(data)
print("\n=== 최종 수집 결과 ===")
print(df_supplementary.head())